# Natural Domains Arithmetic — task demo

A walkthrough of the three things you need before running any analysis on this task:

1. **Causal model** — what variables exist and how they relate.
2. **Templates & token positions** — how prompts get rendered and where the variable-bearing tokens land.
3. **Counterfactual generators** — what `(input, counterfactual)` pairs the analyses see.

This notebook does **not** run interventions. For those, see the demos under `causalab/analyses/<name>/demo.ipynb`. To make the demo cheap and offline-friendly, tokenization uses `gpt2`; swap in any HF model if you want to see model-specific tokenization.

We exercise two domains side-by-side: **`weekdays`** (cyclic, modulus 7) and **`alphabet`** (non-cyclic with `input_filter` and a custom `compute_result`).

In [1]:
from causalab.tasks.natural_domains_arithmetic import (
    DOMAIN_PRESETS,
    NaturalDomainConfig,
    create_causal_model,
    create_token_positions,
    generate_dataset,
)

weekdays = create_causal_model(NaturalDomainConfig(domain_type="weekdays"))
alphabet = create_causal_model(NaturalDomainConfig(domain_type="alphabet"))
weekdays.id, alphabet.id

('natural_domains_arithmetic_weekdays', 'natural_domains_arithmetic_alphabet')

## 1. Causal model

The DAG is identical for every domain:

```
number ──┐
         ├──> result ──> raw_output
entity ──┤
         └──> raw_input
```

Sample one input per domain and trace it forward to see how `entity + number` becomes `result` and `raw_output`.

In [2]:
import random

random.seed(0)

for label, model in [("weekdays", weekdays), ("alphabet", alphabet)]:
    trace = model.sample_input()  # already fully traced (descendants computed)
    print(f"--- {label} ---")
    print(f"  entity     = {trace['entity']!r}")
    print(f"  number     = {trace['number']!r}")
    print(f"  result     = {trace['result']!r}")
    print(f"  raw_input  = {trace['raw_input']!r}")
    print(f"  raw_output = {trace['raw_output']!r}")
    print()

--- weekdays ---
  entity     = 'Sunday'
  number     = 'four'
  result     = 'Thursday'
  raw_input  = 'Q: What day is four days after Sunday?\nA:'
  raw_output = ' Thursday'

--- alphabet ---
  entity     = 'B'
  number     = 'two'
  result     = 'D'
  raw_input  = 'The letter two after B in the alphabet is the letter'
  raw_output = ' D'



Notice the difference in `compute_result`:

- **`weekdays`** wraps with the modulus: `(entity_idx + number) % 7`.
- **`alphabet`** uses a preset-supplied closure (`chr(ord(entity) + number_to_int[number])`) and an `input_filter` that drops pairs whose result would fall outside `D…Z`.

Inspect a few preset fields to confirm:

In [3]:
for name in ["weekdays", "alphabet"]:
    cfg = NaturalDomainConfig(domain_type=name)
    print(f"{name}: cyclic={cfg.cyclic}, modulus={cfg.modulus}, "
          f"|entities|={len(cfg.entities)}, |numbers|={len(cfg.numbers)}, "
          f"result_entities={'<= entities>' if cfg.result_entities is None else len(cfg.result_entities)}")

weekdays: cyclic=True, modulus=7, |entities|=7, |numbers|=7, result_entities=<= entities>
alphabet: cyclic=False, modulus=26, |entities|=25, |numbers|=3, result_entities=23


## 2. Templates & token positions

`create_token_positions(pipeline, template=...)` returns a dict of named `TokenPosition` objects keyed by `last_token`, `entity`, and `number`. Each one is a callable indexer that returns a list of token indices for a given input sample.

Below we tokenize one weekdays prompt with `gpt2` and visualize where each named position lands.

In [4]:
from causalab.neural.pipeline import LMPipeline

pipeline = LMPipeline("gpt2", max_new_tokens=1)
weekdays_template = NaturalDomainConfig(domain_type="weekdays").template
positions = create_token_positions(pipeline, template=weekdays_template)
list(positions.keys())

`torch_dtype` is deprecated! Use `dtype` instead!


['last_token', 'entity', 'number']

In [5]:
sample = weekdays.sample_input()
ids = pipeline.load([sample])["input_ids"][0].tolist()
decoded = [pipeline.tokenizer.decode([t]) for t in ids]

print("Prompt:", sample["raw_input"])
print()
print(f"{'idx':>4}  {'token':<20}  positions")
for i, tok in enumerate(decoded):
    hits = [name for name, pos in positions.items() if i in pos.index(sample)]
    marker = ", ".join(hits) if hits else ""
    print(f"{i:>4}  {tok!r:<20}  {marker}")

Prompt: Q: What day is four days after Friday?
A:

 idx  token                 positions
   0  'Q'                   
   1  ':'                   
   2  ' What'               
   3  ' day'                
   4  ' is'                 
   5  ' four'               number
   6  ' days'               
   7  ' after'              
   8  ' Friday'             entity
   9  '?'                   
  10  '\n'                  
  11  'A'                   
  12  ':'                   last_token


`highlight_selected_token` renders the same information as a single string:

In [6]:
for name, pos in positions.items():
    print(f"[{name}]")
    print("  " + pos.highlight_selected_token(sample))
    print()

[last_token]
  Q: What day is four days after Friday?
A**:**

[entity]
  Q: What day is four days after** Friday**?
A:

[number]
  Q: What day is** four** days after Friday?
A:



For multi-template configs, pass `templates=[...]`; the returned `TokenPosition` objects dispatch on `input_sample["template"]` automatically. The single-template path used above is sufficient for every preset in `DOMAIN_PRESETS`.

## 3. Counterfactual generators

`generate_dataset(model, n, seed)` returns examples of shape `{"input": ..., "counterfactual_inputs": [...]}`. Both base and counterfactual are independent samples — every input variable may differ. Single-variable counterfactuals (only one variable resampled) are configured at the runner level via `task.resample_variable: <var>`; see `ARCHITECTURE.md` §7.

In [7]:
examples = generate_dataset(weekdays, n=3, seed=0)

for i, ex in enumerate(examples):
    base = ex["input"]
    cf = ex["counterfactual_inputs"][0]
    print(f"--- pair {i} ---")
    print(f"  base : entity={base['entity']:<10} number={base['number']:<10} -> result={base['result']}")
    print(f"  cf   : entity={cf['entity']:<10}   number={cf['number']:<10}   -> result={cf['result']}")
    differs = [v for v in ("entity", "number") if base[v] != cf[v]]
    print(f"  differs in: {differs}")
    print()

--- pair 0 ---
  base : entity=Sunday     number=four       -> result=Thursday
  cf   : entity=Sunday       number=four         -> result=Thursday
  differs in: []

--- pair 1 ---
  base : entity=Monday     number=three      -> result=Thursday
  cf   : entity=Friday       number=four         -> result=Tuesday
  differs in: ['entity', 'number']

--- pair 2 ---
  base : entity=Thursday   number=seven      -> result=Thursday
  cf   : entity=Sunday       number=three        -> result=Wednesday
  differs in: ['entity', 'number']



Same call works for any domain. For `alphabet`, the `input_filter` set in `causal_models.py` ensures both base and counterfactual stay inside the reachable result range (`D`…`Z`).

In [8]:
for ex in generate_dataset(alphabet, n=3, seed=0):
    base = ex["input"]
    cf = ex["counterfactual_inputs"][0]
    print(f"  base: {base['entity']} + {base['number']:<6} -> {base['result']}    "
          f"cf: {cf['entity']} + {cf['number']:<6} -> {cf['result']}")

  base: M + two    -> O    cf: B + two    -> D
  base: Q + two    -> S    cf: M + two    -> O
  base: P + two    -> R    cf: S + one    -> T


## Next steps

Once the task is understood, run an analysis end-to-end via Hydra:

```bash
./scripts/run_exp.sh weekdays_8b_baseline   # accuracy + counterfactual sanity
./scripts/run_exp.sh weekdays_8b_pipeline   # baseline → locate → subspace
```

Outputs land under `artifacts/natural_domains_arithmetic/<model>/<analysis>/...`.